In [1]:
import os, json

DATA_DIR = "/kaggle/input/datasets/pradhyumnapawale/golden-test-for-lora-fintech-small-llm"
print("Files found:", os.listdir(DATA_DIR))

GOLDEN_PATH = os.path.join(DATA_DIR, "golden_test.jsonl")
with open(GOLDEN_PATH, "r", encoding="utf-8") as f:
    golden = [json.loads(line) for line in f]

print(f"Loaded {len(golden)} examples. First SMS:\n", golden[0]["sms"])

Files found: ['golden_test.jsonl']
Loaded 30 examples. First SMS:
 Dear UPI user A/C X4821 debited by 245.50 on date 18Nov26 trf to ZOMATO Refno 632189451234. If not u? call 1800111109 -SBI


In [2]:
from kaggle_secrets import UserSecretsClient
key = UserSecretsClient().get_secret("GEMINI_API_KEY")
print("Key loaded successfully! Starts with:", key[:6])

Key loaded successfully! Starts with: AIzaSy


In [3]:
import google.genai as genai
print("SDK ready!")

SDK ready!


In [4]:
import time
from google import genai
from kaggle_secrets import UserSecretsClient

client = genai.Client(api_key=UserSecretsClient().get_secret("GEMINI_API_KEY"))

CANDIDATES = [
    "gemini-3.5-flash-lite",
    "gemini-3.1-flash-lite",
    "gemini-3.6-flash",
    "gemini-3.7-flash",
    "gemini-3.8-flash",
    "gemini-flash-latest",
]

for m in CANDIDATES:
    print(f"Testing {m}...", end=" ", flush=True)
    t0 = time.time()
    try:
        resp = client.models.generate_content(model=m, contents="Say OK")
        print(f"WORKING in {time.time()-t0:.1f}s -> {resp.text.strip()}", flush=True)
        WORKING_MODEL = m
        break
    except Exception as e:
        print(f"Failed ({str(e)[:60]}...)", flush=True)

Testing gemini-3.5-flash-lite... WORKING in 0.6s -> OK


In [5]:
import json
import time
from google import genai
from google.genai import types
from kaggle_secrets import UserSecretsClient

client = genai.Client(api_key=UserSecretsClient().get_secret("GEMINI_API_KEY"))
MODEL = "gemini-3.5-flash-lite"

REQUIRED_KEYS = {
    "is_transaction", "type", "amount",
    "merchant", "payment_mode", "account_last4", "date"
}

SCENARIOS = [
    "UPI debit transactions from SBI, HDFC, ICICI, Axis, Kotak, PNB, Canara, BOI, Union Bank to varied Indian merchants (Swiggy, Zomato, Zepto, Blinkit, Uber, Ola, IRCTC, local kirana, VPA IDs, petrol pumps). Vary date formats (18Nov26, 18-11-2026, 18/11/26, or missing date).",
    "UPI credits, IMPS/NEFT salary or friend transfers, and merchant refunds (Amazon, Flipkart, Myntra, Swiggy) to bank accounts or cards.",
    "Credit card and Debit card swipes/online spends, plus ATM cash withdrawals across different Indian banks. Include some with missing merchant names or commas in amounts (e.g. Rs.12,450.50).",
    "NetBanking transfers, SIP mutual fund auto-debits, EMI deductions, UPI AutoPay mandates (Netflix, Spotify, JioFiber, Airtel).",
    "NON-TRANSACTION TRAPS: OTP messages with amounts, FAILED UPI transactions, insufficient balance alerts, credit card bill due reminders, available balance SMS, pre-approved loan spam, and cashback promo SMS. For ALL of these, is_transaction MUST be false and all other 6 fields MUST be null."
]

PROMPT_TEMPLATE = """Generate 15 realistic, diverse Indian bank/UPI SMS alerts and their exact extracted JSON for this scenario:
{scenario}

STRICT RULES:
1. Return a JSON array of 15 objects, each with exactly two keys: "sms" (string) and "expected" (object).
2. "expected" MUST have these exact 7 keys:
   - "is_transaction": boolean (true only if money actually moved; false for OTPs, failed txns, promos, reminders, balance checks)
   - "type": "debit" | "credit" | null
   - "amount": float/number (never a string, no commas) | null
   - "merchant": string (exact merchant/person/VPA from SMS) | null
   - "payment_mode": "UPI" | "Card" | "NetBanking" | "ATM" | null
   - "account_last4": 4-digit string (e.g. "4821") | null
   - "date": "YYYY-MM-DD" string | null (if date is not in SMS, use null)
3. If "is_transaction" is false, ALL other 6 fields MUST be null.
4. Make the SMS text realistic, messy, and varied (different banks, abbreviations like A/c, Acct, Rs., INR, Refno, Avl Bal).
"""

def is_valid_item(item):
    if not isinstance(item, dict) or "sms" not in item or "expected" not in item:
        return False
    exp = item["expected"]
    if not isinstance(exp, dict) or set(exp.keys()) != REQUIRED_KEYS:
        return False
    if exp["is_transaction"] is False:
        return all(exp[k] is None for k in REQUIRED_KEYS if k != "is_transaction")
    if exp["is_transaction"] is True:
        if exp["type"] not in ("debit", "credit"):
            return False
        if not isinstance(exp["amount"], (int, float)):
            return False
    return True

all_valid_examples = []
seen_sms = set()

# 7 rounds x 5 scenarios x 15 examples = ~525 examples
ROUNDS = 7

for r in range(ROUNDS):
    for idx, scenario in enumerate(SCENARIOS):
        t0 = time.time()
        try:
            response = client.models.generate_content(
                model=MODEL,
                contents=PROMPT_TEMPLATE.format(scenario=scenario),
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=0.9,
                ),
            )
            batch = json.loads(response.text)
            added = 0
            for item in batch:
                if is_valid_item(item) and item["sms"] not in seen_sms:
                    seen_sms.add(item["sms"])
                    all_valid_examples.append(item)
                    added += 1

            # Save progress immediately after every call
            with open("train_raw.jsonl", "w", encoding="utf-8") as f:
                for ex in all_valid_examples:
                    f.write(json.dumps(ex) + "\n")

            print(f"Round {r+1}/{ROUNDS} | Scenario {idx+1}/5 -> +{added} in {time.time()-t0:.1f}s (Total: {len(all_valid_examples)})", flush=True)

        except Exception as e:
            print(f"Round {r+1} Scenario {idx+1} error: {str(e)[:80]}", flush=True)

        time.sleep(3)

print(f"\nDONE! Saved {len(all_valid_examples)} clean training examples to train_raw.jsonl", flush=True)

Round 1/7 | Scenario 1/5 -> +15 in 4.8s (Total: 15)
Round 1/7 | Scenario 2/5 -> +15 in 5.5s (Total: 30)
Round 1/7 | Scenario 3/5 -> +14 in 5.2s (Total: 44)
Round 1/7 | Scenario 4/5 -> +14 in 5.6s (Total: 58)
Round 1/7 | Scenario 5/5 -> +15 in 4.9s (Total: 73)
Round 2/7 | Scenario 1/5 -> +15 in 5.6s (Total: 88)
Round 2/7 | Scenario 2/5 -> +15 in 5.7s (Total: 103)
Round 2/7 | Scenario 3/5 -> +15 in 5.1s (Total: 118)
Round 2/7 | Scenario 4/5 -> +15 in 5.3s (Total: 133)
Round 2/7 | Scenario 5/5 -> +15 in 4.5s (Total: 148)
Round 3/7 | Scenario 1/5 -> +14 in 5.4s (Total: 162)
Round 3/7 | Scenario 2/5 -> +15 in 6.2s (Total: 177)
Round 3/7 | Scenario 3/5 -> +15 in 5.0s (Total: 192)
Round 3/7 | Scenario 4/5 -> +15 in 5.3s (Total: 207)
Round 3/7 | Scenario 5/5 -> +15 in 4.5s (Total: 222)
Round 4/7 | Scenario 1/5 -> +15 in 4.9s (Total: 237)
Round 4/7 | Scenario 2/5 -> +15 in 5.8s (Total: 252)
Round 4/7 | Scenario 3/5 -> +15 in 5.2s (Total: 267)
Round 4/7 | Scenario 4/5 -> +15 in 5.4s (Total: 282)